<a href="https://colab.research.google.com/github/joudData/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joudData/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
### Proposed Baseline Rule

Prioritize published content that has not been updated recently and still has search demand. The goal is to identify content that may benefit from a refresh.

**Signals used:**
- Content staleness: time since the last update.
- Search volume: estimated demand for the content's keyword.

**Reason code:** `REFRESH_STALE_HIGH_DEMAND`

**Action label:** `Refresh Content`

The thresholds will be selected after checking the two signals in the data.

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:


# Combine structural deviation with search demand

import os
import numpy as np
import pandas as pd

# 1. Prepare structural features and search volume
features = [
    "word_count",
    "char_count",
    "keyword_token_count",
    "category_count"
]

baseline_df = content_df[
    ["content_hash_id", "content_type"] + features + ["search_volume"]
].copy()

baseline_df = baseline_df.dropna(subset=["content_type"])

# 2. Impute missing structural values within content type
for col in features:
    baseline_df[col] = baseline_df.groupby(
        "content_type"
    )[col].transform(
        lambda x: x.fillna(x.median())
    )

# 3. Calculate percentile ranks within each content type
deviation_columns = []

for col in features:
    percentile_col = f"{col}_percentile"
    deviation_col = f"{col}_deviation"

    baseline_df[percentile_col] = baseline_df.groupby(
        "content_type"
    )[col].rank(pct=True)

    baseline_df[deviation_col] = (
        (baseline_df[percentile_col] - 0.5).abs() * 2
    )

    deviation_columns.append(deviation_col)

# 4. Calculate structural deviation
baseline_df["structural_deviation"] = (
    baseline_df[deviation_columns].mean(axis=1)
)

# 5. Calculate final score using search demand
baseline_df["score"] = (
    baseline_df["structural_deviation"]
    * np.log1p(baseline_df["search_volume"].clip(lower=0))
)

# 6. Keep valid positive-demand opportunities
queue = baseline_df.dropna(
    subset=["search_volume", "score"]
).copy()

queue = queue[queue["search_volume"] > 0].copy()

# 7. Assign one reason code and one action label
queue["reason_code"] = "HIGH_DEMAND_UNUSUAL_STRUCTURE"

queue["action_label"] = "Review Content Archetype"

# 8. Sort and create rank
queue = queue.sort_values(
    "score", ascending=False
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

# 9. Save the ranked queue
output_path = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(output_path, index=False)

print("Queue created:", len(queue))
print("Missing scores:", queue["score"].isna().sum())
print("Saved:", output_path)

display(queue.head(10))




Queue created: 213353
Missing scores: 0
Saved: work/outputs/baseline_action_score.csv


,rank,content_hash_id,content_type,word_count,char_count,keyword_token_count,category_count,search_volume,word_count_percentile,word_count_deviation,char_count_percentile,char_count_deviation,keyword_token_count_percentile,keyword_token_count_deviation,category_count_percentile,category_count_deviation,structural_deviation,score,reason_code,action_label
0,1,content_04e4047dc8eef2fd,keyword article,4913.0,33207.0,4,9,368000.0,0.982776,0.965551,0.982314,0.964628,0.257588,0.484825,0.992012,0.984023,0.849757,10.890349,HIGH_DEMAND_UNUSUAL_STRUCTURE,Review Content Archetype
1,2,content_004d12f1771eb1f5,keyword article,5065.0,36022.0,3,6,110000.0,0.984572,0.969145,0.987364,0.974729,0.136628,0.726744,0.949327,0.898655,0.892318,10.358246,HIGH_DEMAND_UNUSUAL_STRUCTURE,Review Content Archetype
2,3,content_a7e7140044d3d849,keyword article,3432.0,21052.0,3,7,301000.0,0.919225,0.838449,0.881402,0.762805,0.136628,0.726744,0.970092,0.940184,0.817045,10.306920,HIGH_DEMAND_UNUSUAL_STRUCTURE,Review Content Archetype
3,4,content_00b7e8a53200f31a,keyword article,3389.0,23717.0,3,7,201000.0,0.913572,0.827144,0.940601,0.881202,0.136628,0.726744,0.970092,0.940184,0.843818,10.303922,HIGH_DEMAND_UNUSUAL_STRUCTURE,Review Content Archetype
4,5,content_427fa2debbfdca60,keyword article,3313.0,20218.0,3,9,301000.0,0.902606,0.805211,0.849803,0.699606,0.136628,0.726744,0.992012,0.984023,0.803896,10.141046,HIGH_DEMAND_UNUSUAL_STRUCTURE,Review Content Archetype
5,6,content_237a74ec2680908b,keyword article,3333.0,20481.0,3,7,301000.0,0.905634,0.811268,0.860632,0.721265,0.136628,0.726744,0.970092,0.940184,0.799865,10.090193,HIGH_DEMAND_UNUSUAL_STRUCTURE,Review Content Archetype
6,7,content_ce2918eedb1a0318,keyword article,4117.0,29570.0,2,6,49500.0,0.969042,0.938084,0.974783,0.949566,0.054367,0.891266,0.949327,0.898655,0.919393,9.938404,HIGH_DEMAND_UNUSUAL_STRUCTURE,Review Content Archetype
7,8,content_a218926555b8fb80,keyword article,3164.0,19799.0,3,7,301000.0,0.875268,0.750537,0.831172,0.662344,0.136628,0.726744,0.970092,0.940184,0.769952,9.712845,HIGH_DEMAND_UNUSUAL_STRUCTURE,Review Content Archetype
8,9,content_f6d70715c2c5b2d3,keyword article,3129.0,19429.0,3,7,301000.0,0.867270,0.734541,0.813252,0.626503,0.136628,0.726744,0.970092,0.940184,0.756993,9.549367,HIGH_DEMAND_UNUSUAL_STRUCTURE,Review Content Archetype
9,10,content_07f8a651c96ff872,keyword article,3123.0,19287.0,3,7,301000.0,0.865870,0.731740,0.806006,0.612012,0.136628,0.726744,0.970092,0.940184,0.752670,9.494833,HIGH_DEMAND_UNUSUAL_STRUCTURE,Review Content Archetype


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:

top_10 = queue.head(10)

for _, row in top_10.iterrows():
    print(
        f"Rank {row['rank']}: {row['content_hash_id']} | "
        f"Action: {row['action_label']} | "
        f"Why: High search volume ({row['search_volume']:,.0f}) "
        f"and unusual structure "
        f"(deviation={row['structural_deviation']:.2f}). | "
        f"Could be wrong if: The unusual structure is intentional "
        f"and appropriate for its content type."
    )


Rank 1: content_04e4047dc8eef2fd | Action: Review Content Archetype | Why: High search volume (368,000) and unusual structure (deviation=0.85). | Could be wrong if: The unusual structure is intentional and appropriate for its content type.
Rank 2: content_004d12f1771eb1f5 | Action: Review Content Archetype | Why: High search volume (110,000) and unusual structure (deviation=0.89). | Could be wrong if: The unusual structure is intentional and appropriate for its content type.
Rank 3: content_a7e7140044d3d849 | Action: Review Content Archetype | Why: High search volume (301,000) and unusual structure (deviation=0.82). | Could be wrong if: The unusual structure is intentional and appropriate for its content type.
Rank 4: content_00b7e8a53200f31a | Action: Review Content Archetype | Why: High search volume (201,000) and unusual structure (deviation=0.84). | Could be wrong if: The unusual structure is intentional and appropriate for its content type.
Rank 5: content_427fa2debbfdca60 | Actio

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:

# Review the lowest-ranked picks

weak_picks = queue.tail(5).sort_values("score")

for _, row in weak_picks.iterrows():
    print(
        f"{row['content_hash_id']} | "
        f"Score: {row['score']:.4f} | "
        f"Search volume: {row['search_volume']:,.0f} | "
        f"Structural deviation: {row['structural_deviation']:.2f} | "
        f"Potential weakness: Low priority because its combined "
        f"demand and structural deviation score is low."
    )


content_b419f0d6e789ba33 | Score: 0.2228 | Search volume: 10 | Structural deviation: 0.09 | Potential weakness: Low priority because its combined demand and structural deviation score is low.
content_b3dfb59c74a5864d | Score: 0.2228 | Search volume: 10 | Structural deviation: 0.09 | Potential weakness: Low priority because its combined demand and structural deviation score is low.
content_b3ed4986596fdd75 | Score: 0.2228 | Search volume: 10 | Structural deviation: 0.09 | Potential weakness: Low priority because its combined demand and structural deviation score is low.
content_b44cc586292ed87f | Score: 0.2228 | Search volume: 10 | Structural deviation: 0.09 | Potential weakness: Low priority because its combined demand and structural deviation score is low.
content_b3bc63651e25a948 | Score: 0.2228 | Search volume: 10 | Structural deviation: 0.09 | Potential weakness: Low priority because its combined demand and structural deviation score is low.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.